# PREPROCESSING PIPELINE

In [ ]:
import numpy as np
import pandas as pd
import pyreadr

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

## 1. LOAD RDS FILE

In [ ]:
RDS_PATH = "data/features_all_windows_combined.rds"

result = pyreadr.read_r(RDS_PATH)
df = next(iter(result.values()))
df = pd.DataFrame(df)

## 2. REMOVE ACCIDENTAL INDEX / UNNAMED COLUMNS

In [ ]:
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

## 3. TIME FEATURE ENGINEERING

In [ ]:
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

df["session_duration_sec"] = (
    df["end_time"] - df["start_time"]
).dt.total_seconds().clip(lower=1)

hour = df["start_time"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

dow = df["start_time"].dt.weekday
df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
df["dow_cos"] = np.cos(2 * np.pi * dow / 7)

df = df.drop(columns=["start_time", "end_time"])

## 4. BINARY FEATURES (KEEP AS 0/1)

In [ ]:
binary_cols = [
    "gps_home", "gps_work",
    "headset_PLUGGED", "headset_UNPLUGGED",
    "phone_AIRPLANE", "phone_BATTERYSAVINGMODE",
    "phone_BLUETOOTH", "phone_CAMERA", "phone_CALENDAR",
    "phone_PHONE", "phone_POWER", "phone_SMS"
]

binary_cols = [c for c in binary_cols if c in df.columns]

for col in binary_cols:
    df[col] = (df[col] > 0).astype(int)

## 5. MUSIC KEY AS CATEGORICAL


In [ ]:
if "music_track_key" in df.columns:
    df["music_track_key"] = (
        df["music_track_key"]
        .round()
        .astype("Int64")
        .astype("category")
    )

## 6. COLLAPSE RARE CATEGORICAL LEVELS

In [ ]:
def collapse_rare(series, min_freq=0.0001):
    freq = series.value_counts(normalize=True)
    rare = freq[freq < min_freq].index
    return series.where(~series.isin(rare), other="Other")

for col in ["gps_landuse_type", "time_bin"]:
    if col in df.columns:
        df[col] = collapse_rare(df[col])


## 7. DEFINE FEATURE GROUPS


In [ ]:
# user_id stays as-is
pass_through_features = ["user_id"]

categorical_features = [
    "time_bin",
    "gps_landuse_type",
    "music_track_key"
]
categorical_features = [c for c in categorical_features if c in df.columns]

# numeric features exclude user_id and binary columns
numeric_features = [
    c for c in df.columns
    if c not in categorical_features + binary_cols + pass_through_features
]

## 8. PREPROCESSING PIPELINES

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

binary_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("bin", binary_pipeline, binary_cols),
        ("cat", categorical_pipeline, categorical_features),
        ("pass", "passthrough", pass_through_features)  # <- user_id untouched
    ]
)

## 9. FIT + TRANSFORM

In [ ]:
X = preprocessor.fit_transform(df)

# 10. FEATURE NAMES

In [ ]:
feature_names = preprocessor.get_feature_names_out()
X = pd.DataFrame(X, columns=feature_names)

print("Preprocessing complete.")
print(f"Final feature matrix shape: {X.shape}")
print("\nFirst 5 rows:")
print(X.head())

## 11. SAVE OUTPUT

In [ ]:
OUTPUT_CSV = "data/preprocessed_features.csv"
X.to_csv(OUTPUT_CSV, index=False)

print(f"Preprocessed data saved to {OUTPUT_CSV}")